# Destabilizing Mutation → Protein Abundance Analysis\n\nTests whether destabilizing missense mutations reduce protein abundance using shared peptides from psm.tsv.\n\n**Logic:**  \nFor each destabilizing mutation (high AlphaMissense OR SPURS ddG) in protein X within a plex:\n1. Identify which TMT channels carry the mutation (GDC UUID → missense table)\n2. Find **shared peptides**: reference PSMs mapping to protein X (not -mut, not -comp)\n3. Compare their TMT intensities: mutant-carrier channels vs non-carrier channels in same plex\n4. `delta = log2(mut_channel / mean(WT_channels))` — negative = less abundant\n\nA **neutral control** (benign by AM + SPURS) runs identically as negative control."

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP      = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META     = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
MISSENSE     = "/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv"
RESULTS_BASE = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/results"
PLEX_LIST    = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
OUT_DIR      = "/scratch/leduc.an/AAS_Evo/ANALYSIS/destabilization"

# Destabilizing thresholds
AM_THRESHOLD    = 0.564
SPURS_THRESHOLD = 0.5
VAF_THRESHOLD   = 0.3
GNOMAD_MAX      = 0.01

# Neutral control thresholds
AM_BENIGN_MAX    = 0.34
SPURS_BENIGN_MAX = 0.0

N_PLEXES = None   # None = all; set int to test on subset e.g. 5

CHANNEL_ORDER = ["126","127N","127C","128N","128C","129N","129C","130N","130C","131N","131C"]
TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
    "tmt_126c":"126C","tmt_134n":"134N",
}

os.makedirs(OUT_DIR, exist_ok=True)
print("Config loaded")

In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
if N_PLEXES:
    plex_ids = plex_ids[:N_PLEXES]

print(f"Plexes: {len(plex_ids)} | TMT map: {len(tmt):,} rows | GDC meta: {len(gdc):,} rows")

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print("Loading missense table...")
missense = pd.read_csv(MISSENSE, sep="\t", low_memory=False)
missense["am_pathogenicity"] = pd.to_numeric(missense["am_pathogenicity"], errors="coerce")
missense["spurs_ddg"]        = pd.to_numeric(missense["spurs_ddg"], errors="coerce")
missense["VAF"]              = pd.to_numeric(missense["VAF"], errors="coerce")
missense["gnomADe_AF"]       = pd.to_numeric(missense["gnomADe_AF"], errors="coerce").fillna(0)
print(f"Total rows: {len(missense):,}")

base_ok = (missense["VAF"] >= VAF_THRESHOLD) & (missense["gnomADe_AF"] < GNOMAD_MAX)

destab = missense[((missense["am_pathogenicity"] >= AM_THRESHOLD) |
                   (missense["spurs_ddg"] >= SPURS_THRESHOLD)) & base_ok].copy()

neutral = missense[(missense["am_pathogenicity"] <= AM_BENIGN_MAX) &
                   (missense["spurs_ddg"] <= SPURS_BENIGN_MAX) &
                   missense["spurs_ddg"].notna() & base_ok].copy()

# ── SINGLE-MUTATION FILTER ────────────────────────────────────────────────────
# For each (sample_id, gene), count how many distinct missense mutations exist.
# Only analyze proteins where the sample has exactly one mutation in that gene,
# removing confounding from samples with multiple simultaneous mutations.
mut_counts_per_gene = missense.groupby(["sample_id", "SYMBOL"])["HGVSp"].nunique()
single_mut_set = set(zip(
    mut_counts_per_gene[mut_counts_per_gene == 1].index.get_level_values("sample_id"),
    mut_counts_per_gene[mut_counts_per_gene == 1].index.get_level_values("SYMBOL"),
))
print(f"Single-mutation (sample, gene) pairs: {len(single_mut_set):,}")

# Filter destab/neutral to single-mutation pairs only
destab_single  = destab[destab.apply(lambda r: (r["sample_id"], r["SYMBOL"]) in single_mut_set, axis=1)]
neutral_single = neutral[neutral.apply(lambda r: (r["sample_id"], r["SYMBOL"]) in single_mut_set, axis=1)]
print(f"Destabilizing after single-mut filter: {len(destab_single):,} rows")
print(f"Neutral after single-mut filter:       {len(neutral_single):,} rows")

# sample_id (GDC UUID) -> set of gene symbols with (single) mutations
destab_sample_genes  = destab_single.groupby("sample_id")["SYMBOL"].apply(set).to_dict()
neutral_sample_genes = neutral_single.groupby("sample_id")["SYMBOL"].apply(set).to_dict()

print(f"\nDestabilizing: {len(destab_single):,} rows, {destab_single['SYMBOL'].nunique()} genes, {len(destab_sample_genes)} samples")
print(f"Neutral:       {len(neutral_single):,} rows, {neutral_single['SYMBOL'].nunique()} genes, {len(neutral_sample_genes)} samples")

In [ ]:
# ── HELPERS ───────────────────────────────────────────────────────────────────
def find_ri_cols(df):
    """Returns dict channel_label -> column_name for TMT intensity columns."""
    found = {}
    all_channels = CHANNEL_ORDER + ["126C","132N","132C","133N","133C","134N"]
    for ch in all_channels:
        if ch in df.columns:
            found[ch] = ch; continue
        candidates = [c for c in df.columns if c.startswith("Intensity") and c.endswith(f"_{ch}")]
        if candidates:
            found[ch] = candidates[0]
    return found

def get_channel_uuid_map(plex_id):
    """Returns dict: TMT channel label -> gdc_file_id."""
    pt = tmt[tmt["run_metadata_id"] == plex_id][["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates()
    pt = pt[~pt["case_submitter_id"].str.lower().isin(["ref","reference","pooled","pool","nan"])]
    pt["channel"] = pt["tmt_channel"].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=["channel"])
    merged = pt.merge(gdc[["gdc_file_id","case_submitter_id","sample_type"]],
                      on=["case_submitter_id","sample_type"], how="left")
    return merged.dropna(subset=["gdc_file_id"]).drop_duplicates("channel").set_index("channel")["gdc_file_id"].to_dict()

def gene_from_entry(entry_name):
    """Gene symbol from PSM Entry Name (GENE_HUMAN format). Returns None for -mut/-comp."""
    if pd.isna(entry_name):
        return None
    s = str(entry_name)
    if s.endswith(("-mut", "-comp")):
        return None
    m = re.match(r'^([A-Z0-9]+)_HUMAN', s)
    return m.group(1) if m else None

print("Helpers defined")

In [ ]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
# For each plex, load psm.tsv.
# For each reference PSM (not -mut/-comp), get gene from Entry Name.
# For channels carrying a destabilizing/neutral mutation in that gene:
#   delta = log2(mut_channel_intensity / mean(WT_channel_intensities))
# Only includes (sample, gene) pairs with exactly one mutation (no confounders).

def run_loop(sample_gene_map, label="destabilizing"):
    records = []
    n_done = 0

    for plex_id in plex_ids:
        results_dir = os.path.join(RESULTS_BASE, plex_id)
        psm_files = sorted(glob.glob(os.path.join(results_dir, "*_1/psm.tsv")))
        if not psm_files:
            psm_files = sorted(glob.glob(os.path.join(results_dir, "psm.tsv")))
        if not psm_files:
            continue

        try:
            psm = pd.read_csv(psm_files[0], sep="\t", low_memory=False)
        except Exception:
            continue

        ri_col_map = find_ri_cols(psm)
        if not ri_col_map:
            continue

        ch_uuid = get_channel_uuid_map(plex_id)
        ri_channels = list(ri_col_map.keys())
        plex_chs = [ch for ch in ri_channels if ch in ch_uuid]

        # Reference PSMs only
        ref_mask = ~psm["Entry Name"].astype(str).str.endswith(("-mut", "-comp"))
        ref_psm = psm[ref_mask].copy()
        ref_psm["_gene"] = ref_psm["Entry Name"].apply(gene_from_entry)
        ref_psm = ref_psm.dropna(subset=["_gene"])

        for _, row in ref_psm.iterrows():
            gene = row["_gene"]

            # Channels in this plex that carry a mutation in this gene,
            # AND pass the single-mutation filter (no other mutations on same protein)
            mut_chs = [ch for ch in plex_chs
                       if gene in sample_gene_map.get(ch_uuid.get(ch, ""), set())
                       and (ch_uuid.get(ch, ""), gene) in single_mut_set]
            if not mut_chs:
                continue

            # WT channels = in plex but no mutation in this gene
            wt_chs = [ch for ch in plex_chs
                      if ch not in mut_chs
                      and gene not in sample_gene_map.get(ch_uuid.get(ch, ""), set())]
            if len(wt_chs) < 2:
                continue

            wt_vals = [row[ri_col_map[ch]] for ch in wt_chs
                       if pd.notna(row[ri_col_map[ch]]) and row[ri_col_map[ch]] > 0]
            if len(wt_vals) < 2:
                continue
            wt_mean = np.mean(wt_vals)

            for mut_ch in mut_chs:
                v = row[ri_col_map[mut_ch]]
                if pd.isna(v) or v <= 0:
                    continue
                records.append({
                    "plex_id":   plex_id,
                    "gene":      gene,
                    "channel":   mut_ch,
                    "sample_id": ch_uuid.get(mut_ch, ""),
                    "peptide":   row.get("Peptide", ""),
                    "delta":     np.log2(v / wt_mean),
                    "wt_n":      len(wt_vals),
                })

        n_done += 1
        if n_done % 10 == 0:
            print(f"  [{label}] {n_done} plexes, {len(records):,} records")

    print(f"[{label}] Done: {n_done} plexes, {len(records):,} records")
    return pd.DataFrame(records)

print("Running destabilizing loop...")
df = run_loop(destab_sample_genes, "destabilizing")
print("\nRunning neutral loop...")
df_neutral = run_loop(neutral_sample_genes, "neutral")

In [ ]:
# ── GLOBAL STATISTICS ─────────────────────────────────────────────────────────
t,   p   = stats.ttest_1samp(df["delta"].dropna(),         popmean=0)
t_n, p_n = stats.ttest_1samp(df_neutral["delta"].dropna(), popmean=0)
u,   p_mw = stats.mannwhitneyu(df["delta"].dropna(), df_neutral["delta"].dropna(), alternative="less")

print("── PSM-LEVEL ──────────────────────────────────────────────────────────")
print(f"Destabilizing: n={len(df):,}  mean={df['delta'].mean():.4f}  t={t:.2f}  p={p:.2e}")
print(f"Neutral:       n={len(df_neutral):,}  mean={df_neutral['delta'].mean():.4f}  t={t_n:.2f}  p={p_n:.2e}")
print(f"Mann-Whitney U (destabilizing < neutral): p={p_mw:.2e}")

# ── MUTATION COUNTS (unique sample–gene pairs contributing data) ───────────────
if len(df):
    n_mut_pairs_destab = df[["sample_id","gene"]].drop_duplicates().shape[0]
    n_mut_genes_destab = df["gene"].nunique()
    n_mut_plexes_destab = df["plex_id"].nunique()
else:
    n_mut_pairs_destab = n_mut_genes_destab = n_mut_plexes_destab = 0

if len(df_neutral):
    n_mut_pairs_neutral = df_neutral[["sample_id","gene"]].drop_duplicates().shape[0]
    n_mut_genes_neutral = df_neutral["gene"].nunique()
    n_mut_plexes_neutral = df_neutral["plex_id"].nunique()
else:
    n_mut_pairs_neutral = n_mut_genes_neutral = n_mut_plexes_neutral = 0

print()
print("── MUTATION COUNTS CONTRIBUTING TO ANALYSIS ──────────────────────────")
print(f"Destabilizing: {n_mut_pairs_destab:,} unique (sample, gene) pairs | "
      f"{n_mut_genes_destab} genes | {n_mut_plexes_destab} plexes")
print(f"Neutral:       {n_mut_pairs_neutral:,} unique (sample, gene) pairs | "
      f"{n_mut_genes_neutral} genes | {n_mut_plexes_neutral} plexes")

In [ ]:
# ── COLLAPSE TO PROTEIN LEVEL ─────────────────────────────────────────────────
# Average PSM deltas across all shared peptides for each (plex, gene, channel)
# → one data point per protein per mutant sample observation

def collapse_protein(df):
    return (df.groupby(["plex_id", "gene", "channel"])["delta"]
              .mean()
              .reset_index()
              .rename(columns={"delta": "mean_delta"}))

prot_destab  = collapse_protein(df)
prot_neutral = collapse_protein(df_neutral)

t_p,   p_p   = stats.ttest_1samp(prot_destab["mean_delta"].dropna(),  popmean=0)
t_pn,  p_pn  = stats.ttest_1samp(prot_neutral["mean_delta"].dropna(), popmean=0)
u_p, p_mw_p  = stats.mannwhitneyu(prot_destab["mean_delta"].dropna(),
                                   prot_neutral["mean_delta"].dropna(), alternative="less")

print(f"Protein-level destabilizing: n={len(prot_destab):,}  mean={prot_destab['mean_delta'].mean():.4f}  t={t_p:.2f}  p={p_p:.2e}")
print(f"Protein-level neutral:       n={len(prot_neutral):,}  mean={prot_neutral['mean_delta'].mean():.4f}  t={t_pn:.2f}  p={p_pn:.2e}")
print(f"Mann-Whitney U (destabilizing < neutral): p={p_mw_p:.2e}")

In [ ]:
# ── PER-PLEX HISTOGRAMS ───────────────────────────────────────────────────────
# One subplot per TMT multiplexed set, showing destabilizing (red) and
# neutral (green) protein-level delta distributions side by side.
from matplotlib.backends.backend_pdf import PdfPages

MIN_N_PER_PLEX = 3   # skip plexes with fewer protein observations than this

all_plexes = sorted(set(prot_destab["plex_id"].tolist() + prot_neutral["plex_id"].tolist()))
plexes_to_plot = [p for p in all_plexes
                  if (len(prot_destab[prot_destab["plex_id"] == p]) >= MIN_N_PER_PLEX or
                      len(prot_neutral[prot_neutral["plex_id"] == p]) >= MIN_N_PER_PLEX)]
print(f"Plexes with data (≥{MIN_N_PER_PLEX} observations): {len(plexes_to_plot)}")

NCOLS = 5
nrows = -(-len(plexes_to_plot) // NCOLS)   # ceiling division

fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 3.5, nrows * 2.8))
axes_flat = axes.flatten() if nrows > 1 else list(axes) if NCOLS > 1 else [axes]

# Compute shared x-axis limits from 1st/99th percentiles across all data
all_deltas = pd.concat([prot_destab["mean_delta"], prot_neutral["mean_delta"]]).dropna()
xlim = max(abs(all_deltas.quantile(0.01)), abs(all_deltas.quantile(0.99)))
xlim = min(xlim, 5.0)
bins = np.linspace(-xlim, xlim, 25)

for ax, plex_id in zip(axes_flat, plexes_to_plot):
    d = prot_destab[prot_destab["plex_id"] == plex_id]["mean_delta"].dropna()
    n = prot_neutral[prot_neutral["plex_id"] == plex_id]["mean_delta"].dropna()

    if len(d):
        ax.hist(d, bins=bins, color="#d62728", alpha=0.65, label=f"destab n={len(d)}")
        ax.axvline(d.mean(), color="#d62728", lw=1.2, ls="-")
    if len(n):
        ax.hist(n, bins=bins, color="#2ca02c", alpha=0.65, label=f"neutral n={len(n)}")
        ax.axvline(n.mean(), color="#2ca02c", lw=1.2, ls="-")

    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_xlim(-xlim, xlim)

    # Short label: tissue + date from plex ID (e.g. CCRCC_JHU_20171007)
    parts = plex_id.split("_")
    short = "_".join(parts[2:5]) if len(parts) >= 5 else plex_id[:22]
    ax.set_title(short, fontsize=5.5, pad=2)
    ax.tick_params(labelsize=5)
    ax.legend(fontsize=4.5, loc="upper right", framealpha=0.5)
    ax.set_xlabel("mean log2 delta", fontsize=5)

for ax in axes_flat[len(plexes_to_plot):]:
    ax.set_visible(False)

plt.suptitle(f"Per-plex protein-level delta: destabilizing vs neutral\n"
             f"(only proteins with single mutation in sample, ≥{MIN_N_PER_PLEX} obs per plex)",
             fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_plex_histograms.pdf"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved per-plex histogram PDF: {OUT_DIR}/per_plex_histograms.pdf")

In [ ]:
# ── PLOT: side-by-side protein-level delta distributions ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

q01 = min(prot_destab["mean_delta"].quantile(0.01), prot_neutral["mean_delta"].quantile(0.01))
q99 = max(prot_destab["mean_delta"].quantile(0.99), prot_neutral["mean_delta"].quantile(0.99))

for ax, data, label, color, t_val, p_val in [
    (axes[0], prot_destab,  f"Destabilizing\n(AM≥{AM_THRESHOLD} OR SPURS≥{SPURS_THRESHOLD})", "#d62728", t_p,  p_p),
    (axes[1], prot_neutral, f"Neutral\n(AM≤{AM_BENIGN_MAX} AND SPURS≤{SPURS_BENIGN_MAX})",    "#2ca02c", t_pn, p_pn),
]:
    ax.hist(data["mean_delta"], bins=80, color=color, alpha=0.75, edgecolor="none", range=(q01, q99))
    mean_d = data["mean_delta"].mean()
    ax.axvline(0, color="k", lw=1, ls="--")
    ax.axvline(mean_d, color="k", lw=1.5, ls="-",
               label=f"mean={mean_d:.3f}\nt={t_val:.2f}, p={p_val:.1e}")
    ax.set_xlabel("protein-level mean log2(mut channel / plex WT mean)")
    ax.set_ylabel("protein observations")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle(f"Protein abundance: destabilizing vs neutral mutations (protein-level)\nMann-Whitney p={p_mw_p:.2e}", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "destab_vs_neutral_dist_protein.pdf"), dpi=150)
plt.show()

In [ ]:
# ── PER-GENE STATS (destabilizing, protein-level) ─────────────────────────────
from statsmodels.stats.multitest import multipletests

gene_stats = []
for gene, grp in prot_destab.groupby("gene"):
    deltas = grp["mean_delta"].dropna().values
    if len(deltas) < 3:
        continue
    t_g, p_g = stats.ttest_1samp(deltas, popmean=0)
    gene_stats.append({"gene": gene, "n_proteins": len(deltas), "n_plexes": grp["plex_id"].nunique(),
                       "mean_delta": np.mean(deltas), "t_stat": t_g, "p_value": p_g})

gene_stats = pd.DataFrame(gene_stats).sort_values("p_value")
if len(gene_stats):
    _, gene_stats["fdr"], _, _ = multipletests(gene_stats["p_value"], method="fdr_bh")

sig = (gene_stats["fdr"] < 0.05) & (gene_stats["mean_delta"] < 0)
print(f"Genes tested: {len(gene_stats)} | Significant (FDR<0.05, delta<0): {sig.sum()}")
gene_stats.head(20)

In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(OUT_DIR, "destab_per_psm.tsv"), sep="\t", index=False)
df_neutral.to_csv(os.path.join(OUT_DIR, "neutral_per_psm.tsv"), sep="\t", index=False)
prot_destab.to_csv(os.path.join(OUT_DIR, "destab_per_protein.tsv"), sep="\t", index=False)
prot_neutral.to_csv(os.path.join(OUT_DIR, "neutral_per_protein.tsv"), sep="\t", index=False)
gene_stats.to_csv(os.path.join(OUT_DIR, "gene_destab_stats.tsv"), sep="\t", index=False)
print(f"Saved to {OUT_DIR}")